# Section 07 lab — Guided e-commerce pipeline

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Stage 1 — Define schemas and inspect reference data

Prepare the explicit contracts and small dimensions used by all three runs.


### Step 1 — Set project paths and imports

Keep the project paths short and visible.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from IPython.display import HTML, display
from decimal import Decimal
import json, os
PROJECT = 'project'
os.makedirs(f'{PROJECT}/reports', exist_ok=True)


### Step 2 — Declare the three schemas

Landing transactions remain strings; reference columns have stable names.


In [ ]:
transaction_schema = StructType([
    StructField('transaction_id', StringType()), StructField('transaction_date', StringType()),
    StructField('customer_id', StringType()), StructField('product_id', StringType()),
    StructField('quantity', StringType()), StructField('unit_price', StringType()),
    StructField('status', StringType())])
customer_schema = StructType([
    StructField('customer_id', StringType()), StructField('customer_name', StringType()), StructField('city', StringType())])
product_schema = StructType([
    StructField('product_id', StringType()), StructField('product_name', StringType()),
    StructField('category', StringType()), StructField('list_price', StringType())])


### Step 3 — Read and verify the references

The dimensions must be unique before they can safely enrich transactions.


In [ ]:
customers = spark.read.schema(customer_schema).option('header', True).csv(f'{PROJECT}/reference/customers.csv')
products = spark.read.schema(product_schema).json(f'{PROJECT}/reference/products.json')
assert customers.groupBy('customer_id').count().filter('count > 1').count() == 0
assert products.groupBy('product_id').count().filter('count > 1').count() == 0
customers.show()
products.show()


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
customers.createOrReplaceTempView('project_customers')
products.createOrReplaceTempView('project_products')
spark.sql('SELECT customer_id, COUNT(*) AS rows FROM project_customers GROUP BY customer_id HAVING COUNT(*) > 1').show()
spark.sql('SELECT product_id, COUNT(*) AS rows FROM project_products GROUP BY product_id HAVING COUNT(*) > 1').show()


## Stage 2 — run-001 — ingest, validate, enrich and publish

Process the CSV batch using operations introduced in Sections 02–06.


### Step 1 — Read and inspect the batch

Use the explicit transaction schema and the reader for this run format.


In [ ]:
run_id = 'run-001'
source_path = f'{PROJECT}/source/run-001/transactions.csv'
raw = spark.read.schema(transaction_schema).option('header', True).csv(source_path)
raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
raw.show(truncate=False)
print('Raw rows:', raw.count())


### Step 2 — Deduplicate, convert and classify

Safe conversion and rejection reasons preserve every data-quality outcome.


In [ ]:
unique = raw.dropDuplicates(['transaction_id'])
parsed = (unique.withColumn('transaction_id', F.trim('transaction_id'))
          .withColumn('customer_id', F.trim('customer_id'))
          .withColumn('product_id', F.trim('product_id'))
          .withColumn('status', F.upper(F.trim('status')))
          .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
          .withColumn('qty', F.expr('try_cast(quantity as int)'))
          .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
quality_reason = F.concat_ws('; ',
    F.when(F.col('transaction_id').isNull() | (F.length('transaction_id') == 0), 'missing transaction_id'),
    F.when(F.col('order_date').isNull(), 'invalid transaction_date'),
    F.when(F.col('customer_id').isNull() | (F.length('customer_id') == 0), 'missing customer_id'),
    F.when(F.col('product_id').isNull() | (F.length('product_id') == 0), 'missing product_id'),
    F.when(F.col('qty').isNull() | (F.col('qty') <= 0), 'quantity must be positive'),
    F.when(F.col('price').isNull() | (F.col('price') < 0), 'unit_price must be non-negative'),
    F.when(F.col('status').isNull() | ~F.col('status').isin('COMPLETED', 'CANCELLED', 'RETURNED'), 'unknown status'))
classified = parsed.withColumn('reject_reason', quality_reason)
quality_rejected = classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache()
valid = (classified.filter(F.col('reject_reason') == '').drop('reject_reason').withColumn('line_total', F.col('qty') * F.col('price')))
quality_rejected.select('transaction_id', 'reject_reason').show(truncate=False)
print('Unique:', unique.count(), 'Valid:', valid.count(), 'Quality rejected:', quality_rejected.count())


### Step 3 — Join customer and product references

Left joins expose unknown keys before accepted rows are selected.


In [ ]:
completed = valid.filter(F.col('status') == 'COMPLETED')
joined = (completed.join(F.broadcast(customers), 'customer_id', 'left').join(F.broadcast(products), 'product_id', 'left'))
reference_reason = F.concat_ws('; ', F.when(F.col('customer_name').isNull(), 'unknown customer_id'), F.when(F.col('product_name').isNull(), 'unknown product_id'))
reference_rejected = (joined.withColumn('reject_reason', reference_reason).filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('REFERENCE')).cache())
accepted = (joined.filter(F.col('customer_name').isNotNull() & F.col('product_name').isNotNull()).withColumn('order_month', F.date_format('order_date', 'yyyy-MM')).cache())
reference_rejected.select('transaction_id', 'reject_reason').show(truncate=False)
accepted.select('transaction_id', 'customer_name', 'product_name', 'category', 'line_total').show(truncate=False)


### Step 4 — Write outputs and build the run report

Publish Parquet first, then report from the destination that was written.


In [ ]:
rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
curated_path = f'{PROJECT}/destination/curated_sales/run_id={run_id}'
rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
accepted.drop('run_id').write.mode('overwrite').partitionBy('order_month').parquet(curated_path)
rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
revenue = accepted.agg(F.sum('line_total').alias('value')).first()['value'] or Decimal('0')
metrics = {'run_id': run_id, 'raw_rows': raw.count(), 'duplicate_rows': raw.count() - unique.count(), 'unique_rows': unique.count(), 'quality_rejected_rows': quality_rejected.count(), 'valid_rows': valid.count(), 'non_completed_rows': valid.count() - completed.count(), 'reference_rejected_rows': reference_rejected.count(), 'rejected_rows': quality_rejected.count() + reference_rejected.count(), 'accepted_sales_rows': accepted.count(), 'accepted_revenue': f'{revenue:.2f}'}
spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
destination = spark.read.parquet(curated_path)
category_rows = destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).orderBy('category').collect()
largest = max([float(row['revenue']) for row in category_rows] or [1.0])
bars = ''.join(f"<p><b>{row['category']}</b><br><span style='display:inline-block;background:#2878b5;height:18px;width:{float(row['revenue'])/largest*70:.1f}%'></span> {float(row['revenue']):,.2f}</p>" for row in category_rows)
report_html = f"<h2>{run_id} destination report</h2><p>Accepted: <b>{metrics['accepted_sales_rows']}</b> &nbsp; Rejected: <b>{metrics['rejected_rows']}</b> &nbsp; Revenue: <b>{metrics['accepted_revenue']}</b></p>{bars}"
with open(f'{PROJECT}/reports/{run_id}.html', 'w', encoding='utf-8') as stream: stream.write(report_html)
display(HTML(report_html))
print(metrics)
for frame in (raw, quality_rejected, reference_rejected, accepted): frame.unpersist()


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
raw.createOrReplaceTempView('run_001_raw')
customers.createOrReplaceTempView('project_customers')
products.createOrReplaceTempView('project_products')
sql_result = spark.sql("""
WITH unique_rows AS (
  SELECT * FROM (SELECT r.*, ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY transaction_id) AS duplicate_position FROM run_001_raw r) WHERE duplicate_position = 1
), typed AS (
  SELECT *, TRY_CAST(transaction_date AS DATE) AS order_date,
         TRY_CAST(quantity AS INT) AS qty,
         TRY_CAST(unit_price AS DECIMAL(12,2)) AS price
  FROM unique_rows
), valid AS (
  SELECT * FROM typed
  WHERE order_date IS NOT NULL AND qty > 0 AND price >= 0
    AND TRIM(customer_id) <> '' AND TRIM(product_id) <> ''
    AND status = 'COMPLETED'
)
SELECT v.transaction_id, v.customer_id, c.customer_name, c.city,
       v.product_id, p.product_name, p.category, v.order_date, v.qty, v.price,
       v.qty * v.price AS line_total
FROM valid v
INNER JOIN project_customers c ON v.customer_id = c.customer_id
INNER JOIN project_products p ON v.product_id = p.product_id
ORDER BY v.transaction_id
""")
sql_result.show(truncate=False)
assert sql_result.count() == destination.count()
assert f"{sql_result.agg({'line_total': 'sum'}).first()[0]:.2f}" == metrics['accepted_revenue']


## Stage 3 — run-002 — ingest, validate, enrich and publish

Process the JSON batch using operations introduced in Sections 02–06.


### Step 1 — Read and inspect the batch

Use the explicit transaction schema and the reader for this run format.


In [ ]:
run_id = 'run-002'
source_path = f'{PROJECT}/source/run-002/transactions.json'
raw = spark.read.schema(transaction_schema).json(source_path)
raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
raw.show(truncate=False)
print('Raw rows:', raw.count())


### Step 2 — Deduplicate, convert and classify

Safe conversion and rejection reasons preserve every data-quality outcome.


In [ ]:
unique = raw.dropDuplicates(['transaction_id'])
parsed = (unique.withColumn('transaction_id', F.trim('transaction_id'))
          .withColumn('customer_id', F.trim('customer_id'))
          .withColumn('product_id', F.trim('product_id'))
          .withColumn('status', F.upper(F.trim('status')))
          .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
          .withColumn('qty', F.expr('try_cast(quantity as int)'))
          .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
quality_reason = F.concat_ws('; ',
    F.when(F.col('transaction_id').isNull() | (F.length('transaction_id') == 0), 'missing transaction_id'),
    F.when(F.col('order_date').isNull(), 'invalid transaction_date'),
    F.when(F.col('customer_id').isNull() | (F.length('customer_id') == 0), 'missing customer_id'),
    F.when(F.col('product_id').isNull() | (F.length('product_id') == 0), 'missing product_id'),
    F.when(F.col('qty').isNull() | (F.col('qty') <= 0), 'quantity must be positive'),
    F.when(F.col('price').isNull() | (F.col('price') < 0), 'unit_price must be non-negative'),
    F.when(F.col('status').isNull() | ~F.col('status').isin('COMPLETED', 'CANCELLED', 'RETURNED'), 'unknown status'))
classified = parsed.withColumn('reject_reason', quality_reason)
quality_rejected = classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache()
valid = (classified.filter(F.col('reject_reason') == '').drop('reject_reason').withColumn('line_total', F.col('qty') * F.col('price')))
quality_rejected.select('transaction_id', 'reject_reason').show(truncate=False)
print('Unique:', unique.count(), 'Valid:', valid.count(), 'Quality rejected:', quality_rejected.count())


### Step 3 — Join customer and product references

Left joins expose unknown keys before accepted rows are selected.


In [ ]:
completed = valid.filter(F.col('status') == 'COMPLETED')
joined = (completed.join(F.broadcast(customers), 'customer_id', 'left').join(F.broadcast(products), 'product_id', 'left'))
reference_reason = F.concat_ws('; ', F.when(F.col('customer_name').isNull(), 'unknown customer_id'), F.when(F.col('product_name').isNull(), 'unknown product_id'))
reference_rejected = (joined.withColumn('reject_reason', reference_reason).filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('REFERENCE')).cache())
accepted = (joined.filter(F.col('customer_name').isNotNull() & F.col('product_name').isNotNull()).withColumn('order_month', F.date_format('order_date', 'yyyy-MM')).cache())
reference_rejected.select('transaction_id', 'reject_reason').show(truncate=False)
accepted.select('transaction_id', 'customer_name', 'product_name', 'category', 'line_total').show(truncate=False)


### Step 4 — Write outputs and build the run report

Publish Parquet first, then report from the destination that was written.


In [ ]:
rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
curated_path = f'{PROJECT}/destination/curated_sales/run_id={run_id}'
rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
accepted.drop('run_id').write.mode('overwrite').partitionBy('order_month').parquet(curated_path)
rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
revenue = accepted.agg(F.sum('line_total').alias('value')).first()['value'] or Decimal('0')
metrics = {'run_id': run_id, 'raw_rows': raw.count(), 'duplicate_rows': raw.count() - unique.count(), 'unique_rows': unique.count(), 'quality_rejected_rows': quality_rejected.count(), 'valid_rows': valid.count(), 'non_completed_rows': valid.count() - completed.count(), 'reference_rejected_rows': reference_rejected.count(), 'rejected_rows': quality_rejected.count() + reference_rejected.count(), 'accepted_sales_rows': accepted.count(), 'accepted_revenue': f'{revenue:.2f}'}
spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
destination = spark.read.parquet(curated_path)
category_rows = destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).orderBy('category').collect()
largest = max([float(row['revenue']) for row in category_rows] or [1.0])
bars = ''.join(f"<p><b>{row['category']}</b><br><span style='display:inline-block;background:#2878b5;height:18px;width:{float(row['revenue'])/largest*70:.1f}%'></span> {float(row['revenue']):,.2f}</p>" for row in category_rows)
report_html = f"<h2>{run_id} destination report</h2><p>Accepted: <b>{metrics['accepted_sales_rows']}</b> &nbsp; Rejected: <b>{metrics['rejected_rows']}</b> &nbsp; Revenue: <b>{metrics['accepted_revenue']}</b></p>{bars}"
with open(f'{PROJECT}/reports/{run_id}.html', 'w', encoding='utf-8') as stream: stream.write(report_html)
display(HTML(report_html))
print(metrics)
for frame in (raw, quality_rejected, reference_rejected, accepted): frame.unpersist()


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
raw.createOrReplaceTempView('run_002_raw')
customers.createOrReplaceTempView('project_customers')
products.createOrReplaceTempView('project_products')
sql_result = spark.sql("""
WITH unique_rows AS (
  SELECT * FROM (SELECT r.*, ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY transaction_id) AS duplicate_position FROM run_002_raw r) WHERE duplicate_position = 1
), typed AS (
  SELECT *, TRY_CAST(transaction_date AS DATE) AS order_date,
         TRY_CAST(quantity AS INT) AS qty,
         TRY_CAST(unit_price AS DECIMAL(12,2)) AS price
  FROM unique_rows
), valid AS (
  SELECT * FROM typed
  WHERE order_date IS NOT NULL AND qty > 0 AND price >= 0
    AND TRIM(customer_id) <> '' AND TRIM(product_id) <> ''
    AND status = 'COMPLETED'
)
SELECT v.transaction_id, v.customer_id, c.customer_name, c.city,
       v.product_id, p.product_name, p.category, v.order_date, v.qty, v.price,
       v.qty * v.price AS line_total
FROM valid v
INNER JOIN project_customers c ON v.customer_id = c.customer_id
INNER JOIN project_products p ON v.product_id = p.product_id
ORDER BY v.transaction_id
""")
sql_result.show(truncate=False)
assert sql_result.count() == destination.count()
assert f"{sql_result.agg({'line_total': 'sum'}).first()[0]:.2f}" == metrics['accepted_revenue']


## Stage 4 — run-003 — ingest, validate, enrich and publish

Process the TSV batch using operations introduced in Sections 02–06.


### Step 1 — Read and inspect the batch

Use the explicit transaction schema and the reader for this run format.


In [ ]:
run_id = 'run-003'
source_path = f'{PROJECT}/source/run-003/transactions.tsv'
raw = spark.read.schema(transaction_schema).option('header', True).option('sep', '\t').csv(source_path)
raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
raw.show(truncate=False)
print('Raw rows:', raw.count())


### Step 2 — Deduplicate, convert and classify

Safe conversion and rejection reasons preserve every data-quality outcome.


In [ ]:
unique = raw.dropDuplicates(['transaction_id'])
parsed = (unique.withColumn('transaction_id', F.trim('transaction_id'))
          .withColumn('customer_id', F.trim('customer_id'))
          .withColumn('product_id', F.trim('product_id'))
          .withColumn('status', F.upper(F.trim('status')))
          .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
          .withColumn('qty', F.expr('try_cast(quantity as int)'))
          .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
quality_reason = F.concat_ws('; ',
    F.when(F.col('transaction_id').isNull() | (F.length('transaction_id') == 0), 'missing transaction_id'),
    F.when(F.col('order_date').isNull(), 'invalid transaction_date'),
    F.when(F.col('customer_id').isNull() | (F.length('customer_id') == 0), 'missing customer_id'),
    F.when(F.col('product_id').isNull() | (F.length('product_id') == 0), 'missing product_id'),
    F.when(F.col('qty').isNull() | (F.col('qty') <= 0), 'quantity must be positive'),
    F.when(F.col('price').isNull() | (F.col('price') < 0), 'unit_price must be non-negative'),
    F.when(F.col('status').isNull() | ~F.col('status').isin('COMPLETED', 'CANCELLED', 'RETURNED'), 'unknown status'))
classified = parsed.withColumn('reject_reason', quality_reason)
quality_rejected = classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache()
valid = (classified.filter(F.col('reject_reason') == '').drop('reject_reason').withColumn('line_total', F.col('qty') * F.col('price')))
quality_rejected.select('transaction_id', 'reject_reason').show(truncate=False)
print('Unique:', unique.count(), 'Valid:', valid.count(), 'Quality rejected:', quality_rejected.count())


### Step 3 — Join customer and product references

Left joins expose unknown keys before accepted rows are selected.


In [ ]:
completed = valid.filter(F.col('status') == 'COMPLETED')
joined = (completed.join(F.broadcast(customers), 'customer_id', 'left').join(F.broadcast(products), 'product_id', 'left'))
reference_reason = F.concat_ws('; ', F.when(F.col('customer_name').isNull(), 'unknown customer_id'), F.when(F.col('product_name').isNull(), 'unknown product_id'))
reference_rejected = (joined.withColumn('reject_reason', reference_reason).filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('REFERENCE')).cache())
accepted = (joined.filter(F.col('customer_name').isNotNull() & F.col('product_name').isNotNull()).withColumn('order_month', F.date_format('order_date', 'yyyy-MM')).cache())
reference_rejected.select('transaction_id', 'reject_reason').show(truncate=False)
accepted.select('transaction_id', 'customer_name', 'product_name', 'category', 'line_total').show(truncate=False)


### Step 4 — Write outputs and build the run report

Publish Parquet first, then report from the destination that was written.


In [ ]:
rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
curated_path = f'{PROJECT}/destination/curated_sales/run_id={run_id}'
rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
accepted.drop('run_id').write.mode('overwrite').partitionBy('order_month').parquet(curated_path)
rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
revenue = accepted.agg(F.sum('line_total').alias('value')).first()['value'] or Decimal('0')
metrics = {'run_id': run_id, 'raw_rows': raw.count(), 'duplicate_rows': raw.count() - unique.count(), 'unique_rows': unique.count(), 'quality_rejected_rows': quality_rejected.count(), 'valid_rows': valid.count(), 'non_completed_rows': valid.count() - completed.count(), 'reference_rejected_rows': reference_rejected.count(), 'rejected_rows': quality_rejected.count() + reference_rejected.count(), 'accepted_sales_rows': accepted.count(), 'accepted_revenue': f'{revenue:.2f}'}
spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
destination = spark.read.parquet(curated_path)
category_rows = destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).orderBy('category').collect()
largest = max([float(row['revenue']) for row in category_rows] or [1.0])
bars = ''.join(f"<p><b>{row['category']}</b><br><span style='display:inline-block;background:#2878b5;height:18px;width:{float(row['revenue'])/largest*70:.1f}%'></span> {float(row['revenue']):,.2f}</p>" for row in category_rows)
report_html = f"<h2>{run_id} destination report</h2><p>Accepted: <b>{metrics['accepted_sales_rows']}</b> &nbsp; Rejected: <b>{metrics['rejected_rows']}</b> &nbsp; Revenue: <b>{metrics['accepted_revenue']}</b></p>{bars}"
with open(f'{PROJECT}/reports/{run_id}.html', 'w', encoding='utf-8') as stream: stream.write(report_html)
display(HTML(report_html))
print(metrics)
for frame in (raw, quality_rejected, reference_rejected, accepted): frame.unpersist()


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
raw.createOrReplaceTempView('run_003_raw')
customers.createOrReplaceTempView('project_customers')
products.createOrReplaceTempView('project_products')
sql_result = spark.sql("""
WITH unique_rows AS (
  SELECT * FROM (SELECT r.*, ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY transaction_id) AS duplicate_position FROM run_003_raw r) WHERE duplicate_position = 1
), typed AS (
  SELECT *, TRY_CAST(transaction_date AS DATE) AS order_date,
         TRY_CAST(quantity AS INT) AS qty,
         TRY_CAST(unit_price AS DECIMAL(12,2)) AS price
  FROM unique_rows
), valid AS (
  SELECT * FROM typed
  WHERE order_date IS NOT NULL AND qty > 0 AND price >= 0
    AND TRIM(customer_id) <> '' AND TRIM(product_id) <> ''
    AND status = 'COMPLETED'
)
SELECT v.transaction_id, v.customer_id, c.customer_name, c.city,
       v.product_id, p.product_name, p.category, v.order_date, v.qty, v.price,
       v.qty * v.price AS line_total
FROM valid v
INNER JOIN project_customers c ON v.customer_id = c.customer_id
INNER JOIN project_products p ON v.product_id = p.product_id
ORDER BY v.transaction_id
""")
sql_result.show(truncate=False)
assert sql_result.count() == destination.count()
assert f"{sql_result.agg({'line_total': 'sum'}).first()[0]:.2f}" == metrics['accepted_revenue']


## Stage 5 — Validate and combine all destination data

Use the persisted audit and curated Parquet outputs rather than in-memory run variables.


### Step 1 — Validate every audit partition

Expected values provide a deterministic completion check for each run.


In [ ]:
with open(f'{PROJECT}/validation/expected_metrics.json', encoding='utf-8') as stream: expected = json.load(stream)
with open(f'{PROJECT}/run_config.json', encoding='utf-8') as stream: run_ids = [row['run_id'] for row in json.load(stream)['runs']]
for run_id in run_ids:
    actual = spark.read.parquet(f'{PROJECT}/destination/audit/run_id={run_id}').first().asDict()
    for key, value in expected[run_id].items(): assert actual[key] == value, (run_id, key, actual[key], value)
print('All run audits match expected values')


### Step 2 — Read all curated partitions together

The basePath option recovers run_id and order_month from the partition folders.


In [ ]:
curated_base = f'{PROJECT}/destination/curated_sales'
destination = spark.read.option('basePath', curated_base).parquet(curated_base)
destination.groupBy('run_id', 'order_month').count().orderBy('run_id').show()
assert destination.count() == 21


### Step 3 — Build the combined visual report

The report is calculated entirely from saved destination data.


In [ ]:
category_rows = destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).orderBy('category').collect()
total_revenue = destination.agg(F.sum('line_total')).first()[0]
largest = max([float(row['revenue']) for row in category_rows] or [1.0])
bars = ''.join(f"<p><b>{row['category']}</b><br><span style='display:inline-block;background:#2a9d8f;height:18px;width:{float(row['revenue'])/largest*70:.1f}%'></span> {float(row['revenue']):,.2f}</p>" for row in category_rows)
report_html = f"<h2>All-runs report</h2><p>Accepted: <b>{destination.count()}</b> &nbsp; Revenue: <b>{float(total_revenue):,.2f}</b></p>{bars}"
with open(f'{PROJECT}/reports/all-runs.html', 'w', encoding='utf-8') as stream: stream.write(report_html)
display(HTML(report_html))
assert f'{total_revenue:.2f}' == '103813.00'


### Step 4 — Inspect the final plan

Connect the completed pipeline to exchanges and aggregation stages.


In [ ]:
destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).explain('formatted')


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
destination.createOrReplaceTempView('project_curated_sales')
sql_result = spark.sql("""
SELECT category, COUNT(DISTINCT transaction_id) AS orders, SUM(qty) AS units, SUM(line_total) AS revenue
FROM project_curated_sales
GROUP BY category
ORDER BY category
""")
sql_result.show()
assert sql_result.count() == len(category_rows)
